# Point Source Far-Field Reflector Problem — Python Benchmark

This notebook runs the full pipeline of the **Sinkhorn divergence** approach
to the reflector problem (Benamou, Ijzerman, Rukhaia 2020) and produces
visualisations of every intermediate and final result.

**Algorithm summary:**
- Cost function: $c(x,y) = -\log(1 - x \cdot y)$ (Wang 2004 reflector cost)
- Source $\mu$ on upper hemisphere $S_0$, target $\nu$ on lower hemisphere $S_\infty$
- Both discretised on QMC point clouds: $N_K = 16488$ main grid, $N_K^{\rm small}=381$ warm-start grid
- Sinkhorn divergence: $\mathrm{SD}_\varepsilon(\mu,\nu) = \mathrm{OT}_\varepsilon(\mu,\nu) - \tfrac{1}{2}(\mathrm{OT}_\varepsilon(\mu,\mu') + \mathrm{OT}_\varepsilon(\nu',\nu))$
- Multi-scale regularisation schedule from $k=152$ to $k=1024$
- Warm start from small grid via $c$-transform

## 1. Configuration

In [ ]:
import matplotlib
# Use 'Agg' for compatibility in headless environments.
# Change to 'TkAgg', 'Qt5Agg', or 'widget' for interactive display.
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import numpy as np
import os, time

# ── User settings ──────────────────────────────────────────────────────────
BENCHMARK   = 'SquareToCircle'   # 'SquareToCircle' or 'SquareToTwoGaussSide'
CHUNK_SIZE  = 512                # chunk size for matrix operations
OUTPUT_DIR  = None               # None → auto-generated timestamped directory
VERBOSE     = True
# ───────────────────────────────────────────────────────────────────────────

print(f"Benchmark : {BENCHMARK}")
print(f"Chunk size: {CHUNK_SIZE}")

## 2. Run benchmark

In [ ]:
from benchmark import run_benchmark

t_start = time.perf_counter()

results = run_benchmark(
    benchmark=BENCHMARK,
    output_dir=OUTPUT_DIR,
    chunk_size=CHUNK_SIZE,
    verbose=VERBOSE,
)

t_total = time.perf_counter() - t_start
print(f"\nTotal wall-clock time: {t_total:.1f} s")

In [ ]:
# Unpack results for convenience
x            = results['x']             # (NK, 3) source points
y            = results['y']             # (NK, 3) target points
p            = results['p']             # (NK,) normalised source weights
q            = results['q']             # (NK,) normalised target weights
f            = results['f']             # (NK,) corrected f potential
g            = results['g']             # (NK,) corrected g potential
f_id         = results['f_id']          # (NK,) identity potential f
g_id         = results['g_id']          # (NK,) identity potential g
R            = results['R']             # (NK,) reflector scale
Ref          = results['Ref']           # (NK, 3) reflector points
gc           = results['gc']            # (NK,) c-transform of g
fc           = results['fc']            # (NK,) c-transform of f
Refc         = results['Refc']          # (NK, 3) reflector from gc
dif_f        = results['dif_f']         # f - gc
dif_g        = results['dif_g']         # g - fc
x_regular    = results['x_regular']     # (FinalGrid, 3)
Regular_side = results['Regular_side']  # (1025,)
f_regular    = results['f_regular']     # (FinalGrid,)
Ref_regular  = results['Ref_regular']   # (FinalGrid, 3)
push_result  = results['push_result']   # (K, 3) (u, v, density)
total_cost   = results['total_cost']
output_dir   = results['output_dir']

NK = len(x)
print(f"NK = {NK}")
print(f"FinalGrid = {len(x_regular)}")
print(f"Output directory: {output_dir}")

## 3. Source & target distributions

Scatter plots of the source (upper hemisphere, north-pole projection) and
target (lower hemisphere, south-pole projection) distributions.

In [ ]:
from refracter.distributions import stereo_north, stereo_south

# Source: north-pole stereographic projection
u_src, v_src = stereo_north(x)
src_mask = p > 0

# Target: south-pole stereographic projection
u_tgt, v_tgt = stereo_south(y)
tgt_mask = q > 0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
sc = ax.scatter(
    u_src[src_mask], v_src[src_mask],
    c=p[src_mask], cmap='viridis', s=2, alpha=0.7
)
fig.colorbar(sc, ax=ax, label='weight p[i]')
ax.set_title('Source distribution (north-pole projection)')
ax.set_xlabel('u')
ax.set_ylabel('v')
ax.set_aspect('equal')
ax.grid(True, lw=0.5, alpha=0.5)

ax = axes[1]
sc = ax.scatter(
    u_tgt[tgt_mask], v_tgt[tgt_mask],
    c=q[tgt_mask], cmap='plasma', s=2, alpha=0.7
)
fig.colorbar(sc, ax=ax, label='weight q[j]')
ax.set_title('Target distribution (south-pole projection)')
ax.set_xlabel('u')
ax.set_ylabel('v')
ax.set_aspect('equal')
ax.grid(True, lw=0.5, alpha=0.5)

fig.suptitle(f'{BENCHMARK}: Source and Target', fontsize=14)
fig.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_distributions.png')

## 4. Reflector surface

3-D scatter plots of the reflector surface computed two ways:
- Left: from the Sinkhorn divergence potential $f$ (``Ref``)
- Right: from the $c$-transform of $g$ (``Refc``)

In [ ]:
fig = plt.figure(figsize=(14, 6))

# --- Sinkhorn divergence reflector ---
ax1 = fig.add_subplot(121, projection='3d')
mask = src_mask
sc1 = ax1.scatter(
    Ref[mask, 0], Ref[mask, 1], Ref[mask, 2],
    c=R[mask], cmap='RdYlBu_r', s=1, alpha=0.6
)
fig.colorbar(sc1, ax=ax1, label='R = exp(f)', shrink=0.6)
ax1.set_title('Reflector (Sinkhorn divergence)')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')

# --- c-transform reflector ---
ax2 = fig.add_subplot(122, projection='3d')
sc2 = ax2.scatter(
    Refc[mask, 0], Refc[mask, 1], Refc[mask, 2],
    c=np.exp(gc[mask]), cmap='RdYlBu_r', s=1, alpha=0.6
)
fig.colorbar(sc2, ax=ax2, label='exp(gc)', shrink=0.6)
ax2.set_title('Reflector (c-transform of g)')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')

fig.suptitle(f'{BENCHMARK}: Reflector surface', fontsize=14)
fig.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_reflector_3d.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_reflector_3d.png')

## 5. Kantorovich potentials

2×3 grid:
- Top row: $f$, $f_{\rm id}$, $f - f_{\rm id}$ (source side)
- Bottom row: $g$, $g_{\rm id}$, $g - g_{\rm id}$ (target side)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

def _scatter_potential(ax, u, v, vals, mask, title, cmap='RdBu_r'):
    sc = ax.scatter(u[mask], v[mask], c=vals[mask], cmap=cmap, s=2, alpha=0.7)
    plt.colorbar(sc, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('u')
    ax.set_ylabel('v')
    ax.set_aspect('equal')
    ax.grid(True, lw=0.4, alpha=0.4)

# Top row: source potentials (projected via north pole)
_scatter_potential(axes[0, 0], u_src, v_src, f,              src_mask, 'f (corrected)')
_scatter_potential(axes[0, 1], u_src, v_src, f_id,           src_mask, 'f_id (identity)')
_scatter_potential(axes[0, 2], u_src, v_src, f - f_id,       src_mask, 'f - f_id')

# Bottom row: target potentials (projected via south pole)
_scatter_potential(axes[1, 0], u_tgt, v_tgt, g,              tgt_mask, 'g (corrected)',    cmap='PuOr_r')
_scatter_potential(axes[1, 1], u_tgt, v_tgt, g_id,           tgt_mask, 'g_id (identity)',  cmap='PuOr_r')
_scatter_potential(axes[1, 2], u_tgt, v_tgt, g - g_id,       tgt_mask, 'g - g_id',         cmap='PuOr_r')

fig.suptitle(f'{BENCHMARK}: Kantorovich potentials', fontsize=14)
fig.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_potentials.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_potentials.png')

## 6. C-transform residuals

Histograms of the c-transform residuals:
- $f - g^c$ (should be small if $f$ is approximately $c$-concave)
- $g - f^c$ (should be small if $g$ is approximately $c$-concave)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
vals_f = dif_f[src_mask & np.isfinite(dif_f)]
ax.hist(vals_f, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='red', lw=1.5, ls='--')
ax.set_title(r'Residual $f - g^c$ (source side)')
ax.set_xlabel('f[i] - gc[i]')
ax.set_ylabel('count')
ax.grid(True, lw=0.4, alpha=0.4)
ax.text(0.98, 0.97, f'mean={vals_f.mean():.3e}\nstd={vals_f.std():.3e}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

ax = axes[1]
vals_g = dif_g[tgt_mask & np.isfinite(dif_g)]
ax.hist(vals_g, bins=80, color='darkorange', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='red', lw=1.5, ls='--')
ax.set_title(r'Residual $g - f^c$ (target side)')
ax.set_xlabel('g[j] - fc[j]')
ax.set_ylabel('count')
ax.grid(True, lw=0.4, alpha=0.4)
ax.text(0.98, 0.97, f'mean={vals_g.mean():.3e}\nstd={vals_g.std():.3e}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

fig.suptitle(f'{BENCHMARK}: C-transform residuals', fontsize=14)
fig.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ctransform_residuals.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_ctransform_residuals.png')

## 7. Pushforward / ray tracing

Scatter plot of the reflected rays projected onto the south-pole plane,
coloured by source density.  This shows the approximate pushforward of
the source measure under the reflector map.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Push-forward scatter ---
ax = axes[0]
if len(push_result) > 0:
    u_push = push_result[:, 0]
    v_push = push_result[:, 1]
    rho_push = push_result[:, 2]
    sc = ax.scatter(u_push, v_push, c=rho_push, cmap='hot_r', s=1, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='Source density P(x)')
    ax.set_title(f'Push-forward rays (N={len(push_result)})')
else:
    ax.text(0.5, 0.5, 'No valid rays', ha='center', va='center',
            transform=ax.transAxes, fontsize=14)
    ax.set_title('Push-forward rays (none)')

ax.set_xlabel('u (south-pole projection)')
ax.set_ylabel('v (south-pole projection)')
ax.set_aspect('equal')
ax.grid(True, lw=0.4, alpha=0.4)

# --- Target distribution for reference ---
ax = axes[1]
sc = ax.scatter(
    u_tgt[tgt_mask], v_tgt[tgt_mask],
    c=q[tgt_mask], cmap='plasma', s=2, alpha=0.7
)
plt.colorbar(sc, ax=ax, label='weight q[j]')
ax.set_title('Target distribution (reference)')
ax.set_xlabel('u (south-pole projection)')
ax.set_ylabel('v (south-pole projection)')
ax.set_aspect('equal')
ax.grid(True, lw=0.4, alpha=0.4)

fig.suptitle(f'{BENCHMARK}: Pushforward vs. Target', fontsize=14)
fig.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_pushforward.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_pushforward.png')

## 8. Summary statistics

In [ ]:
import textwrap

summary = f"""
======================================================
  BENCHMARK SUMMARY: {BENCHMARK}
======================================================

  Algorithm
  ---------
  Cost function  : c(x,y) = -log(1 - x·y)  (Wang 2004)
  Algorithm      : Sinkhorn divergence
  Regularisation : multi-scale k=152 → 1024 (step=10)

  Problem size
  ------------
  NK (main grid)   : {NK}
  NK_small         : {len(results.get('x', x))}
  FinalGrid        : {len(x_regular)}  (1025 x 1025)
  Source support   : {int((p>0).sum())} points
  Target support   : {int((q>0).sum())} points

  Results
  -------
  Total cost       : {total_cost:.6e}
  R range          : [{R[src_mask].min():.4e}, {R[src_mask].max():.4e}]
  f range          : [{f[src_mask].min():.4e}, {f[src_mask].max():.4e}]
  g range          : [{g[tgt_mask].min():.4e}, {g[tgt_mask].max():.4e}]
  |f-gc|_inf       : {np.max(np.abs(dif_f[src_mask & np.isfinite(dif_f)])):.4e}
  |g-fc|_inf       : {np.max(np.abs(dif_g[tgt_mask & np.isfinite(dif_g)])):.4e}
  Valid push rays  : {len(push_result)} / {len(results.get('push_cloud', x))} push cloud points

  Timing
  ------
  Total elapsed    : {results['elapsed']:.2f} s

  Output
  ------
  Directory        : {output_dir}
======================================================
"""

print(summary)

# Also save summary to file
with open(os.path.join(output_dir, 'summary.txt'), 'w') as fh:
    fh.write(summary)
print(f"Summary saved to {os.path.join(output_dir, 'summary.txt')}")

## Appendix: Verify QMC data loading

In [ ]:
# Quick sanity checks on the loaded data
print("=== Data sanity checks ===")
print(f"x shape    : {x.shape},  dtype={x.dtype}")
print(f"y shape    : {y.shape},  dtype={y.dtype}")
print(f"x norm range: [{np.linalg.norm(x, axis=1).min():.6f}, {np.linalg.norm(x, axis=1).max():.6f}]  (should be ~1)")
print(f"y norm range: [{np.linalg.norm(y, axis=1).min():.6f}, {np.linalg.norm(y, axis=1).max():.6f}]  (should be ~1)")
print(f"x[:,2] min : {x[:,2].min():.4f}  (should be >= 0 for upper hemisphere)")
print(f"y[:,2] max : {y[:,2].max():.4f}  (should be <= 0 for lower hemisphere)")
print(f"sum(p) = {p.sum():.6f}  (should be 1)")
print(f"sum(q) = {q.sum():.6f}  (should be 1)")
print(f"Regular_side range: [{Regular_side.min():.4f}, {Regular_side.max():.4f}]  (should be [-0.6, 0.6])")